In [73]:
import sqlite3
import pandas as pd
import re
import time
import requests
import numpy as np
from sklearn.neighbors import BallTree

# ver todas las columnas y filas de un dataframe
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [74]:
# Cargar datos desde la base de datos SQLite
con = sqlite3.connect("../01_obtener_datos/avisos_gran_concepcion.db")

df_avisos = pd.read_sql_query("SELECT * FROM avisos", con)
df_detalle = pd.read_sql_query("SELECT * FROM avisos_detalle", con)

con.close()

#### Descripción inicial

In [75]:
print("Número de avisos:", df_avisos.shape[0])
print("Número de avisos con detalles:", df_detalle.shape[0])
print("% de avisos con detalles:", round((df_detalle.shape[0] / df_avisos.shape[0]) * 100, 2), "%")

Número de avisos: 1777
Número de avisos con detalles: 439
% de avisos con detalles: 24.7 %


In [76]:
print("Null values in df_avisos:")
print(round((df_avisos.isna().sum() / df_avisos.shape[0]) * 100, 2))
print()
print("==" * 50)
print()
print("Null values in df_detalle:")
print(round((df_detalle.isna().sum() / df_detalle.shape[0]) * 100, 2))

Null values in df_avisos:
id_aviso          0.00
comuna            0.00
tipo_propiedad    0.00
operacion         0.00
titulo            0.00
precio            0.00
moneda            0.00
ubicacion         0.00
dormitorios       1.13
banos             0.39
superficie_m2     0.68
url               0.00
first_seen        0.00
dtype: float64


Null values in df_detalle:
id_aviso                                0.00
url                                     0.00
descripcion                             2.51
fecha_publicacion_texto                18.22
fecha_publicacion_aprox                18.22
superficie_total_m2                     2.05
superficie_util_m2                      0.46
dormitorios                             0.00
banos                                   0.23
estacionamientos                        0.00
antiguedad_anos                        18.00
amoblado                                0.00
admite_mascotas                         0.23
condominio_cerrado                     40.77
f

#### Limpieza preliminar al EDA

In [77]:
# Filtramos las columnas que nos interesan del dataframe df_avisos
df_avisos = df_avisos[["id_aviso", "titulo", "comuna", "tipo_propiedad", "precio", "moneda", "dormitorios", "banos", "superficie_m2"]]

# Filtramos las columnas que nos interesan del dataframe df_detalle
df_detalle = df_detalle[["id_aviso", "estacionamientos", "antiguedad_anos", "admite_mascotas", "condominio_cerrado",  
                         "distancia_centro_comuna_m", "distancia_centro_concepcion_m", "cantidad_paraderos", 
                           "cantidad_jardines_infantiles", "cantidad_colegios", "cantidad_universidades",  "cantidad_plazas", 
                           "cantidad_supermercados", "cantidad_farmacias", "cantidad_centros_comerciales", "cantidad_hospitales",
                             "cantidad_clinicas",  "cantidad_estaciones_metro", "latitud", "longitud", "fecha_publicacion_aprox",
                              "superficie_total_m2", "superficie_util_m2",
                              
                              
                              "bodegas", "gastos_comunes", "estacionamiento_visitas", "solo_familias", "max_habitantes", 
                              "piscina", "quincho", "conserjeria", "ascensor", "piso_unidad", "deptos_por_piso", "barrio" 
                              
                              
                              
                              
                              ]]

df = pd.merge(df_avisos, df_detalle, on="id_aviso", how="left")

In [78]:
df["estacionamiento_visitas"].value_counts()

estacionamiento_visitas
Sí    139
Name: count, dtype: int64

In [79]:
var = ["bodegas", "gastos_comunes", "estacionamiento_visitas", "solo_familias", "max_habitantes", 
       "piscina", "quincho", "conserjeria", "ascensor", "piso_unidad", "deptos_por_piso", "barrio"]

# Reemplazar Si por 1 y No por 0 para las variables var_bol
var_bol = ["estacionamiento_visitas", "solo_familias", "piscina", "quincho", "conserjeria", "ascensor"]
df[var_bol] = df[var_bol].fillna(0)
df[var_bol] = df[var_bol].replace({"Sí": 1})

# Tranformar variables a formato numerico
var_num = ["bodegas", "gastos_comunes", "estacionamiento_visitas", "solo_familias", "max_habitantes", 
       "piscina", "quincho", "conserjeria", "ascensor", "piso_unidad", "deptos_por_piso"]

df[var_num] = df[var_num].apply(pd.to_numeric, errors="coerce")

df[var].describe()

,bodegas,gastos_comunes,estacionamiento_visitas,solo_familias,max_habitantes,piscina,quincho,conserjeria,ascensor,piso_unidad,deptos_por_piso
count,411.000000,380.000000,1777.000000,1777.000000,201.000000,1495.000000,1759.000000,1697.000000,1722.000000,265.000000,187.000000
mean,0.411192,63.444737,0.078222,0.008441,3.522388,0.034114,0.011939,0.177961,0.148084,9.713208,6.893048
std,0.516811,48.960590,0.268596,0.091513,1.720682,0.181582,0.108641,0.382593,0.355286,30.818518,3.277375
min,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000
25%,0.000000,35.000000,0.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,4.000000,4.000000
50%,0.000000,60.000000,0.000000,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,6.000000,6.000000
75%,1.000000,90.000000,0.000000,0.000000,4.000000,0.000000,0.000000,0.000000,0.000000,12.000000,9.500000
max,2.000000,300.000000,1.000000,1.000000,10.000000,1.000000,1.000000,1.000000,1.000000,502.000000,19.000000


In [ ]:
# Rellenamos los valores nulos de las columnas de cantidad de servicios con 0
vars = ["cantidad_paraderos", "cantidad_jardines_infantiles", "cantidad_colegios", 
    "cantidad_universidades", "cantidad_plazas", "cantidad_supermercados", 
    "cantidad_farmacias", "cantidad_centros_comerciales", "cantidad_hospitales", 
    "cantidad_clinicas", "cantidad_estaciones_metro", "estacionamientos"                  ]

df[vars] = df[vars].fillna(0)

In [46]:
# Eliminar registro si tiene nan en estas variables: dormitorios, baños, superficie_m2
df = df.dropna(subset=['dormitorios', 'banos', 'superficie_m2'])

In [ ]:
""" Donde moneda es UF, usar fecha_publicacion_aprox para buscar el precio de la UF en esa fecha con alguna api y convertir el precio a CLP. 
Si no hay fecha_publicacion_aprox, usar la fecha promedio todas las fecha_publicacion_aprox"""   

API_UF_URL = "https://mindicador.cl/api/uf/{fecha}"


def obtener_valor_uf(fecha: pd.Timestamp, reintentos: int = 3, espera: float = 1.0):
    """
    Consulta el valor de la UF para una fecha específica en mindicador.cl
    (API pública y gratuita del Banco Central de Chile / mindicador.cl).
    Devuelve None si no logra obtenerlo tras los reintentos.
    """
    fecha_str = fecha.strftime("%d-%m-%Y")
    url = API_UF_URL.format(fecha=fecha_str)

    for intento in range(reintentos):
        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                datos = resp.json()
                serie = datos.get("serie", [])
                if serie:
                    return serie[0]["valor"]
            return None
        except requests.RequestException:
            time.sleep(espera)

    return None


# ------------------------------------------------------------------
# Paso 0: asegurar que fecha_publicacion_aprox sea datetime
# ------------------------------------------------------------------
df["fecha_publicacion_aprox"] = pd.to_datetime(df["fecha_publicacion_aprox"], errors="coerce")

# ------------------------------------------------------------------
# Paso 1: fecha de respaldo = promedio de todas las fecha_publicacion_aprox
# (se usa solo para las filas en UF que no tengan fecha propia)
# ------------------------------------------------------------------
fecha_promedio = df["fecha_publicacion_aprox"].mean().normalize()
print(f"Fecha promedio usada como respaldo: {fecha_promedio.date()}")

# ------------------------------------------------------------------
# Paso 2: para cada fila en UF, determinar qué fecha usar
# (la propia si existe, si no la fecha promedio)
# ------------------------------------------------------------------
mask_uf = df["moneda"] == "UF"
fechas_a_usar = df.loc[mask_uf, "fecha_publicacion_aprox"].dt.normalize().fillna(fecha_promedio)

# ------------------------------------------------------------------
# Paso 3: consultar la API solo una vez por fecha única (no por fila),
# para no golpear la API de más innecesariamente
# ------------------------------------------------------------------
fechas_unicas = fechas_a_usar.unique()
print(f"Fechas únicas a consultar en la API: {len(fechas_unicas)}")

valor_uf_por_fecha = {}
for fecha in fechas_unicas:
    fecha_ts = pd.Timestamp(fecha)
    valor = obtener_valor_uf(fecha_ts)

    if valor is None:
        print(f"No se pudo obtener UF para {fecha_ts.date()}, se usará la fecha promedio como respaldo")
        valor = obtener_valor_uf(fecha_promedio)

    valor_uf_por_fecha[fecha_ts] = valor
    time.sleep(0.3)  # para no golpear la API de más

# ------------------------------------------------------------------
# Paso 4: convertir a CLP. Las filas que ya estaban en $ quedan igual.
# ------------------------------------------------------------------
df["precio_clp"] = df["precio"].astype(float)

for idx in df.index[mask_uf]:
    fecha = fechas_a_usar.loc[idx]
    valor_uf = valor_uf_por_fecha.get(fecha)

    if valor_uf is not None:
        df.loc[idx, "precio_clp"] = df.loc[idx, "precio"] * valor_uf
    else:
        df.loc[idx, "precio_clp"] = None  # no se pudo obtener ni la fecha propia ni la de respaldo

print("Filas en UF sin conversión posible:", df.loc[mask_uf, "precio_clp"].isna().sum())

In [ ]:
df.drop(["precio", "moneda"], axis=1, inplace=True)

In [ ]:
"""  
Para las columnas de antiguedad_anos, condominio_cerrado en donde haya nan, rellenar buscando 
en la misma base, los valores que tienen otros registros con la misma latitud y longitud. 
Si no hay registros con la misma latitud y longitud, mantener con nan.                           

"""

df['antiguedad_anos'] = df.groupby(['latitud', 'longitud'])['antiguedad_anos'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x))
df['condominio_cerrado'] = df.groupby(['latitud', 'longitud'])['condominio_cerrado'].transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x))

# Para el resto de nan en condominio_cerrado, rellenar con "No"
df['condominio_cerrado'] = df['condominio_cerrado'].fillna('No')

# Para las columnas y admite_mascotas, rellenar con "No" los nan
df[["admite_mascotas"]] = df[["admite_mascotas"]].fillna("No")

In [ ]:
RADIO_METROS = 200
RADIO_TIERRA_M = 6371000

# ------------------------------------------------------------------
# Paso 0: conversión de tipos (vienen como TEXT desde SQLite)
# ------------------------------------------------------------------
df["antiguedad_anos"] = pd.to_numeric(df["antiguedad_anos"], errors="coerce")
df["latitud"] = pd.to_numeric(df["latitud"], errors="coerce")
df["longitud"] = pd.to_numeric(df["longitud"], errors="coerce")


# ------------------------------------------------------------------
# Función de imputación por cercanía (200m, mismo tipo_propiedad)
# ------------------------------------------------------------------
def imputar_por_cercania_200m(df, columna="antiguedad_anos", radio_metros=RADIO_METROS):
    """
    Para cada fila con NaN en `columna`, busca otras filas del mismo
    tipo_propiedad dentro de `radio_metros` (usando latitud/longitud) y
    devuelve la mediana de esos vecinos. Si no hay vecinos, deja el NaN
    (se resuelve en los fallbacks siguientes).
    """
    valores_originales = df[columna].copy()
    resultado = valores_originales.copy()
    radio_rad = radio_metros / RADIO_TIERRA_M

    for _, grupo in df.groupby("tipo_propiedad"):
        tiene_coords = grupo[["latitud", "longitud"]].notna().all(axis=1)
        tiene_valor = valores_originales.loc[grupo.index].notna()

        idx_con_dato = grupo.index[tiene_valor & tiene_coords]
        idx_sin_dato = grupo.index[~tiene_valor & tiene_coords]  # sin coords no se puede buscar vecinos

        if len(idx_con_dato) == 0 or len(idx_sin_dato) == 0:
            continue

        coords_con_dato = np.radians(
            grupo.loc[idx_con_dato, ["latitud", "longitud"]].astype(float).values
        )
        coords_sin_dato = np.radians(
            grupo.loc[idx_sin_dato, ["latitud", "longitud"]].astype(float).values
        )

        arbol = BallTree(coords_con_dato, metric="haversine")
        vecinos_por_fila = arbol.query_radius(coords_sin_dato, r=radio_rad)

        for idx_fila, vecinos in zip(idx_sin_dato, vecinos_por_fila):
            if len(vecinos) > 0:
                valores_vecinos = valores_originales.loc[idx_con_dato].iloc[vecinos]
                resultado.loc[idx_fila] = valores_vecinos.median()

    return resultado


# ------------------------------------------------------------------
# Guardamos los valores originales ANTES de imputar nada.
# Todos los fallbacks se calculan sobre estos valores originales, nunca
# sobre resultados ya imputados en un paso anterior (para no contaminar
# las medianas con estimaciones previas).
# ------------------------------------------------------------------
valores_originales = df["antiguedad_anos"].copy()

# Paso 1: vecinos dentro de 200m, mismo tipo_propiedad
df["antiguedad_anos"] = imputar_por_cercania_200m(df, "antiguedad_anos", RADIO_METROS)

# Paso 2: mediana por tipo_propiedad (calculada desde los datos ORIGINALES)
mediana_por_tipo = df.groupby("tipo_propiedad")["tipo_propiedad"].transform(
    lambda serie: valores_originales.loc[serie.index].median()
)
df["antiguedad_anos"] = df["antiguedad_anos"].fillna(mediana_por_tipo)

# Paso 3: mediana por comuna (calculada desde los datos ORIGINALES)
mediana_por_comuna = df.groupby("comuna")["comuna"].transform(
    lambda serie: valores_originales.loc[serie.index].median()
)
df["antiguedad_anos"] = df["antiguedad_anos"].fillna(mediana_por_comuna)

# Paso 4: mediana global (también desde los datos ORIGINALES)
df["antiguedad_anos"] = df["antiguedad_anos"].fillna(valores_originales.median())

# ------------------------------------------------------------------
# Paso 5: convertir a entero, solo ahora que ya no debería quedar ningún NaN
# ------------------------------------------------------------------
print("NaN restantes antes de convertir:", df["antiguedad_anos"].isna().sum())  # debería ser 0
df["antiguedad_anos"] = df["antiguedad_anos"].round().astype("Int64")

In [ ]:
df.drop(columns=['latitud', 'longitud', "fecha_publicacion_aprox", 
       #          "id_aviso"
                 ], inplace=True)

In [ ]:
# Transformar la columna tipo_propiedad en variables dummy (one-hot encoding), eliminar la columna original y dejar solo la columnas de casas

df = pd.get_dummies(df, columns=["tipo_propiedad"], prefix="tipo_propiedad", drop_first=True)

In [ ]:
# Transformar a variables numéricas las columnas de dormitorios, baños, superficie_m2 y estacionamientos.
vars = ["dormitorios", "banos", "superficie_m2", "estacionamientos"]

df[vars] = df[vars].apply(pd.to_numeric, errors="coerce")

In [ ]:
df["comuna"].value_counts()

In [ ]:
COMUNAS_A_AGRUPAR = [
    "talcahuano-biobio",
    "chiguayante-biobio",
    "hualpen-biobio",
    "coronel-biobio",
    "tome-biobio",
    "penco-biobio",
    "hualqui-biobio",
]

df.loc[df["comuna"].isin(COMUNAS_A_AGRUPAR), "comuna"] = "otras comunas"

In [ ]:
# Comunas convertidas a variables dummy (one-hot encoding), eliminando la columna original y 
# dejando solo las columnas de comunas.
df = pd.get_dummies(df, columns=["comuna"], prefix="comuna", drop_first=True)

In [ ]:
# Cubre las variantes más comunes: amoblado/amoblada, amueblado/amueblada,
# y también sin tilde por si algún título viene mal escrito.
PATRON_AMOBLADO = re.compile(r"amoblad[oa]|amueblad[oa]", re.IGNORECASE)

df["amoblado"] = df["titulo"].str.contains(PATRON_AMOBLADO, na=False).astype(int)
df.drop(columns=["titulo"], inplace=True)

In [ ]:
# si "superficie_total_m2" o "superficie_util_m2" es nan, rellenar con "superficie_m2"
df["superficie_total_m2"] = df["superficie_total_m2"].fillna(df["superficie_m2"])
df["superficie_util_m2"] = df["superficie_util_m2"].fillna(df["superficie_m2"])
# Transformar a numérico las columnas de superficie_total_m2 y superficie_util_m2
df[["superficie_total_m2", "superficie_util_m2"]] = df[["superficie_total_m2", "superficie_util_m2"]].apply(pd.to_numeric, errors="coerce")


df.drop(columns=["superficie_m2"], inplace=True)

In [ ]:
# Si "admite_mascotas" y "condominio_cerrado" son si, dejar como true, si dicen no, dejar como false. 
# Cambiar el tipo de dato a booleano.
df["admite_mascotas"] = df["admite_mascotas"].str.lower().map({"si": True, "no": False})
df["condominio_cerrado"] = df["condominio_cerrado"].str.lower().map({"si": True, "no": False})

#### EDA

In [ ]:
df.isnull().sum()

In [ ]:
df[["id_aviso","precio_clp"]].sort_values("precio_clp", ascending=False).head(15)

In [ ]:
# Eliminar los registros con precio_clp mayor a 8 millones de pesos, ya que los mayores son venta
df = df[df["precio_clp"] <= 8000000]

In [ ]:
df.drop("id_aviso", axis=1, inplace=True)

In [ ]:
df.describe(include='all')

#### Modelo prelimimar de prueba

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ------------------------------------------------------------------
# Separar variable objetivo (y) del resto de columnas (X)
# ------------------------------------------------------------------
y = df["precio_clp"]
X = df.drop(columns=["precio_clp"])

# ------------------------------------------------------------------
# Codificar columnas categóricas (texto) a variables dummy (0/1).
# Random Forest no puede recibir texto directamente.
# ------------------------------------------------------------------
columnas_categoricas = X.select_dtypes(include=["object", "category", "str"]).columns.tolist()
print(f"Columnas categóricas detectadas y codificadas: {columnas_categoricas}")

X = pd.get_dummies(X, columns=columnas_categoricas, drop_first=True)

# ------------------------------------------------------------------
# Split entrenamiento / prueba
# ------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Filas entrenamiento: {len(X_train)} | Filas prueba: {len(X_test)}")

# ------------------------------------------------------------------
# Entrenar el modelo
# ------------------------------------------------------------------
modelo = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
)
modelo.fit(X_train, y_train)

# ------------------------------------------------------------------
# Predecir y calcular métricas de desempeño
# ------------------------------------------------------------------
y_pred = modelo.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mape = (np.abs((y_test - y_pred) / y_test)).mean() * 100

print("\n--- Métricas de desempeño (conjunto de prueba) ---")
print(f"MAE  (error absoluto promedio):      ${mae:,.0f}")
print(f"RMSE (raíz del error cuadrático):     ${rmse:,.0f}")
print(f"R²   (varianza explicada):            {r2:.3f}")
print(f"MAPE (error porcentual promedio):     {mape:.1f}%")

# ------------------------------------------------------------------
# Importancia de variables
# ------------------------------------------------------------------
importancia = pd.DataFrame({
    "variable": X.columns,
    "importancia": modelo.feature_importances_,
}).sort_values("importancia", ascending=False).reset_index(drop=True)

print("\n--- Importancia de variables ---")
print(importancia.to_string(index=False))